In [ ]:
# This file is for running the result of the last query retrieval and last query response
API_KEY = ""

# Choose a gemini model
import google.generativeai as genai
from google.colab import userdata
GOOGLE_API_KEY = API_KEY
genai.configure(api_key=GOOGLE_API_KEY)
gemini_model = genai.GenerativeModel('models/gemini-2.5-flash-lite')

# Load the structurized manual
from google.colab import drive
drive.mount('/content/drive')

file_path = "/content/drive/MyDrive/Haystack/Dataset/manual.json"
import json
with open(file_path, "r", encoding="utf-8") as file:
  manual=json.load(file)

# save the contexts, titles and pages for later use
contexts = [document["context"] for document in manual]
titles = [document["title"] for document in manual]
pages = [document["page"] for document in manual]

Mounted at /content/drive


In [ ]:
!pip install haystack-ai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 624.7/624.7 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.2/145.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.1/80.1 kB 6.9 MB/s eta 0:00:00


In [ ]:
# Prepare the document_store, doc_embedder and text_embedder using haystack
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack import Document
from haystack.components.embedders import SentenceTransformersDocumentEmbedder, SentenceTransformersTextEmbedder
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack.components.builders import PromptBuilder
from haystack.components.generators import HuggingFaceLocalGenerator
from haystack.utils import ComponentDevice

document_store = InMemoryDocumentStore()
docs = [Document(content = contexts[i], meta = {'title': titles[i], 'context_id': i, 'page': pages[i]}) for i in range(len(manual))]

doc_embedder = SentenceTransformersDocumentEmbedder(
    model="all-MiniLM-L6-v2",
    progress_bar=False
)
doc_embedder.warm_up()

docs_with_embeddings = doc_embedder.run(docs)
document_store.write_documents(docs_with_embeddings["documents"])

text_embedder = SentenceTransformersTextEmbedder(
    model="all-MiniLM-L6-v2",
    progress_bar=False
)
text_embedder.warm_up()

retriever = InMemoryEmbeddingRetriever(document_store)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# define a function for rewritting a query
def rewrite_query(query, history_conversations):
  template_for_query_rewrite = f"""
You are an intelligent assistant for rewriting a question.
You need to rewrite a new question based on the history conversations.
-----------------------------------------------------------------------------------
History conversations:
{history_conversations}
-----------------------------------------------------------------------------------
New question: {query}
Rewrite the new question according to the history conversations. Make sure the rewritten new question is concise and understandable without the history conversations.
Rewrite the new question in the format below:
Rewritten Question: ...
"""
  # generate rewritten query by gemini model
  model_rewritten_query = gemini_model.generate_content(template_for_query_rewrite)
  # clean the model's rewritten query
  rewritten_query = model_rewritten_query.text.strip("Rewritten Question:")
  rewritten_query = rewritten_query.strip()
  # return the rewritten query
  #print(template_for_query_rewrite) # debug code
  #print(rewritten_query)
  return rewritten_query

# define a function for generating a prompt for LLM response
def generate_prompt_for_response(query, retrieved_context_ids, contexts, history_queries, history_responses):
  # manage the history conversations
  history_conversations = ""
  if len(history_queries) == 0:
    history_conversations = "\nThere is no history conversation yet."
  else:
    for i in range(len(history_queries)):
      history_conversations += "\n" + f"Question {i+1}: " + history_queries[i]
      history_conversations += "\n" + f"Response {i+1}: " + history_responses[i]

  # convert context ids into the actual context
  retrieved_contexts = ""
  for i in range(len(retrieved_context_ids)):
    retrieved_contexts += f"\n{contexts[retrieved_context_ids[i]]}"

  # put retrieved_contexts and query into the template
  template = f"""
You are an aviation manual's intelligent assistant.
You need to answer a question based on the following context and the history conversations.
-----------------------------------------------------------------------------------
Context:
{retrieved_contexts}
-----------------------------------------------------------------------------------
History Conversations:
{history_conversations}
-----------------------------------------------------------------------------------
New question: {query}
Not every context is useful. Answer the new question only based on the most relevant context.
Read the history conversations as well because they might be useful for you to understand the new question.
The answer should be concise and rigorous. Only provide the core part of the answer.
Answer the question in the format below:
Answer: ...
"""
  return template

In [ ]:
# define a context retriever
def context_retriever(retrieval_input, num_retrieved):
  retrieval_input_embedding = text_embedder.run(retrieval_input)
  docs = retriever.run(query_embedding = retrieval_input_embedding['embedding'], top_k = num_retrieved)
  return [docs["documents"][k].meta['context_id'] for k in range(num_retrieved)]

# this function gives a response according to the query and num_retrieved
def generate_gemini_model_response_from_one_query(query, num_retrieved, history_queries, history_responses):
  # rewrite the last query
  if len(history_queries) == 0:
    rewritten_query = query
  else:
    history_conversations = ""
    for i in range(len(history_queries)):
      history_conversations += "\n" + f"Question {i+1}: " + history_queries[i]
      history_conversations += "\n" + f"Response {i+1}: " + history_responses[i]
    rewritten_query = rewrite_query(query, history_conversations)

  # retrieve context ids using the rewritten query
  retrieved_context_ids = context_retriever(retrieval_input = rewritten_query, num_retrieved = num_retrieved)

  # create the prompt using all conversations and the retrieved context
  prompt = generate_prompt_for_response(query = query, retrieved_context_ids = retrieved_context_ids, contexts = contexts, history_queries = history_queries, history_responses = history_responses)
  # run the result for gemini model using the prompt
  model_response = gemini_model.generate_content(prompt)

  # convert the response into a string
  response = model_response.text.strip("Answer: ")
  response = response.strip()

  # return the reponse
  #print(prompt) ### debug code
  #print(response) ### debug code
  return [response, retrieved_context_ids, rewritten_query]

# this function gives a list of responses, representing responses for one conversation
def generate_gemini_model_responses_from_one_conversation(queries, num_retrieved):
  # it returns [query1_response_and_ids_and_rewritten_query, query2_response_and_ids_and_rewritten_query, ...]
  one_conversation_result = []

  history_queries = []
  history_responses = []
  for query in queries:
    # generate the response and retrieved context ids
    new_response, new_context_ids, new_rewritten_query = generate_gemini_model_response_from_one_query(query, num_retrieved, history_queries, history_responses)
    # update the one_conversation_result
    one_conversation_result.append([new_response, new_context_ids, new_rewritten_query])
    # update the history_queries
    history_queries.append(query)
    # update the history_responses
    history_responses.append(new_response)

  return one_conversation_result

In [ ]:
# load the QAs
manual_QAs_json_path = "/content/drive/MyDrive/Haystack/Dataset/manual_QAs.json"

with open(manual_QAs_json_path, "r", encoding="utf-8") as qas_file:
  manual_QAs = json.load(qas_file)

# load the queries
Queries = {}
Queries["LDS"] = [conversational_QAs['Queries'] for conversational_QAs in manual_QAs["LDS"]]
Queries["ESS"] = [conversational_QAs['Queries'] for conversational_QAs in manual_QAs["ESS"]]
Queries["RNS"] = [conversational_QAs['Queries'] for conversational_QAs in manual_QAs["RNS"]]

In [ ]:
# some parameters
gemini_model_results = {"LDS": {"top_k=1": [],
                                  "top_k=3": [],
                                  "top_k=5": [],
                                  "top_k=10": []},
                          "ESS": {"top_k=1": [],
                                  "top_k=3": [],
                                  "top_k=5": [],
                                  "top_k=10": []},
                          "RNS": {"top_k=1": [],
                                  "top_k=3": [],
                                  "top_k=5": [],
                                  "top_k=10": []}}
top_ks = [1, 3, 5, 10]
sample_types = ["LDS", "ESS", "RNS"]

In [ ]:
# model_result_from_one_conversation = generate_gemini_model_responses_from_one_conversation(queries = Queries["LDS"][0], num_retrieved = 3) ##bedug code

In [ ]:
import time
def result(gemini_model_results, sample_type, top_k):
  count = 0
  for queries in Queries[sample_type]:
    count += 1
    if count%5 == 0:
      print(count)
    if count == 20:
      time.sleep(30) # wait for 30 seconds to buffer and avoid connection error
    try:
      model_result_from_one_conversation = generate_gemini_model_responses_from_one_conversation(queries = queries, num_retrieved = top_k)
    except:
      gemini_model_results[sample_type][f"top_k={top_k}"].append("Error")
    else:
      model_responses_from_one_conversation = [result[0] for result in model_result_from_one_conversation]
      model_retrieved_context_ids_from_one_conversation = [result[1] for result in model_result_from_one_conversation]
      model_rewritten_query_from_one_conversation = [result[2] for result in model_result_from_one_conversation]
      gemini_model_results[sample_type][f"top_k={top_k}"].append({"Responses": model_responses_from_one_conversation,
                                                                  "Retrieved_context_ids": model_retrieved_context_ids_from_one_conversation,
                                                                  "Rewritten_queries": model_rewritten_query_from_one_conversation})
  time.sleep(30)

In [ ]:
result(gemini_model_results = gemini_model_results, sample_type = "LDS", top_k = 1)
result(gemini_model_results = gemini_model_results, sample_type = "ESS", top_k = 1)
result(gemini_model_results = gemini_model_results, sample_type = "RNS", top_k = 1)
# save progress
json_path = "/content/drive/MyDrive/Haystack/Result/result_rewrite_all_1.json"
with open(json_path, 'w') as f:
    json.dump(gemini_model_results, f, indent=4)

5
10
15
20
25
30
35
40
5
10
15
20
25
30
35
40
5
10
15
20
25
30
35
40


In [12]:
json_path = "/content/drive/MyDrive/Haystack/Result/result_rewrite_all_1.json"
with open(json_path, "r", encoding="utf-8") as file:
  gemini_model_results = json.load(file)
result(gemini_model_results = gemini_model_results, sample_type = "LDS", top_k = 3)
result(gemini_model_results = gemini_model_results, sample_type = "ESS", top_k = 3)
result(gemini_model_results = gemini_model_results, sample_type = "RNS", top_k = 3)
# save progress
json_path = "/content/drive/MyDrive/Haystack/Result/result_rewrite_all_1_3.json"
with open(json_path, 'w') as f:
    json.dump(gemini_model_results, f, indent=4)

5
10
15
20
25
30
35
40
5
10
15
20
25
30
35
40
5
10
15
20
25
30
35
40


In [13]:
json_path = "/content/drive/MyDrive/Haystack/Result/result_rewrite_all_1_3.json"
with open(json_path, "r", encoding="utf-8") as file:
  gemini_model_results = json.load(file)
result(gemini_model_results = gemini_model_results, sample_type = "LDS", top_k = 5)
result(gemini_model_results = gemini_model_results, sample_type = "ESS", top_k = 5)
result(gemini_model_results = gemini_model_results, sample_type = "RNS", top_k = 5)
# save progress
json_path = "/content/drive/MyDrive/Haystack/Result/result_rewrite_all_1_3_5.json"
with open(json_path, 'w') as f:
    json.dump(gemini_model_results, f, indent=4)

5
10
15
20
25
30
35
40
5
10
15
20
25
30
35
40
5
10
15
20
25
30
35
40


In [14]:
json_path = "/content/drive/MyDrive/Haystack/Result/result_rewrite_all_1_3_5.json"
with open(json_path, "r", encoding="utf-8") as file:
  gemini_model_results = json.load(file)
result(gemini_model_results = gemini_model_results, sample_type = "LDS", top_k = 10)
result(gemini_model_results = gemini_model_results, sample_type = "ESS", top_k = 10)
result(gemini_model_results = gemini_model_results, sample_type = "RNS", top_k = 10)
# save progress
json_path = "/content/drive/MyDrive/Haystack/Result/result_rewrite_all.json"
with open(json_path, 'w') as f:
    json.dump(gemini_model_results, f, indent=4)

5
10
15
20
25
30
35
40
5
10
15
20
25
30
35
40
5
10
15
20
25
30
35
40
